# StringForge ecosystem pipeline

This notebook demonstrates how the StringForge packages communicate with each
other to form a complete computational pipeline from Calabi–Yau geometry to
axion phenomenology:

1. **JAXVacua** — find flux vacua (complex-structure + axio-dilaton sector)
2. **KahlerJAX** — stabilise the Kähler moduli
3. **JAXiverse** — extract the axion spectrum and cosmological observables

Steps 1 is executable now (public packages). Steps 2–3 use packages that
are not yet publicly released; their code cells are included as documentation
with representative outputs.

## Step 0 — Setup

In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np

## Step 1 — Find flux vacua with JAXVacua

We start by loading a Calabi–Yau model and finding flux vacua in the
complex-structure and axio-dilaton sector.

In [4]:
from jaxvacua import FluxEFT

# Load CP^{11169}[18] at LCS with instanton corrections
model = FluxEFT(h12=2, model_ID=1, maximum_degree=2, Q=276)

# Example flux vector
fluxes = jnp.array([7, 3, -24, 0, -16, 50, 0, 3, -4, 0, 0, 0])

# Known vacuum point
z_vac = jnp.array([2.742j, 2.057j])
tau_vac = 6.855j

# Verify: covariant derivatives should vanish at a vacuum
DW = model.DW(z_vac, jnp.conj(z_vac), tau_vac, jnp.conj(tau_vac), fluxes)
print("|D_i W| =", jnp.abs(DW))

# Extract the vacuum value of the superpotential
W0 = model.superpotential(z_vac, tau_vac, fluxes)
print("W_0 =", W0)

|D_i W| = [0.00104528 0.0080582  0.002     ]
W_0 = (-5.182278073334601e-08+2.842170943040401e-14j)


## Step 2 — Stabilise Kähler moduli with KahlerJAX

```{admonition} Not yet public
:class: warning
KahlerJAX is not yet publicly released. The code below illustrates the
intended API; it will become executable in a future StringForge release.
```

Given the vacuum value $W_0$ from Step 1, we stabilise the Kähler moduli
using non-perturbative effects and $\alpha'$-corrections.

In [ ]:
# --- KahlerJAX (not yet public) ---
#
# from kahlerjax import kahler_sector
#
# kahler_model = kahler_sector(cy=cy, W0=W0)
# kahler_vac = kahler_model.find_minimum(initial_guess=t0)
#
# # Kähler moduli at the vacuum
# t_vac = kahler_vac.kahler_moduli
# V_CY = kahler_vac.cy_volume
# print("Kähler moduli:", t_vac)
# print("CY volume:", V_CY)

## Step 3 — Extract the axion spectrum with JAXiverse

```{admonition} Not yet public
:class: warning
JAXiverse is not yet publicly released. The code below illustrates the
intended API; it will become executable in a future StringForge release.
```

Given the stabilised Kähler moduli from Step 2, we extract the full
multi-axion spectrum: masses, decay constants, and couplings.

In [ ]:
# --- JAXiverse (not yet public) ---
#
# from jaxiverse.geometry import to_eft_spectrum
#
# spectrum = to_eft_spectrum(
#     cy, kahler=t_vac,
#     gs=1/jnp.sqrt(jnp.imag(tau_vac)),
#     W0=jnp.abs(W0),
#     strategy="auto"
# )
#
# print("Axion masses (log10 eV):", spectrum["log10_m_eV"])
# print("Decay constants (log10 GeV):", spectrum["log10_f_gs_GeV"])

## Summary

The full StringForge pipeline connects:

| Step | Package | Input | Output |
|------|---------|-------|--------|
| 1 | JAXVacua | CY geometry + flux quanta | $z_*$, $\tau_*$, $W_0$ |
| 2 | KahlerJAX | $W_0$ + CY topology | $t_*$ (Kähler moduli), $\mathcal{V}$ |
| 3 | JAXiverse | $t_*$, $g_s$, $W_0$ + CY topology | axion masses, decay constants, couplings |

Each step is differentiable (JAX-native), enabling gradient-based
optimisation and sensitivity analysis across the entire pipeline.